**Import required libraries**

In [2]:
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

**Load the dataset**

In [3]:
# Dataset Path
CURRENT_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(CURRENT_DIR)

DATASET_PATH = os.path.join(PROJECT_ROOT,"Datasets","RAVDESS")

print("Dataset Path:", DATASET_PATH)

Dataset Path: C:\Users\yuvra\AGE_GENDER_DETECTION\Datasets\RAVDESS


**Extract Labels from Filename**

In [4]:
# Load Dataset Information
file_paths = []
genders = []
emotions = []
ages = []

for root, dirs, files in os.walk(DATASET_PATH):

    for file in files:

        if file.endswith(".wav"):

            path = os.path.join(root,file)

            parts = file.split("-")

            emotion = int(parts[2])
            actor = int(parts[-1].split(".")[0])

            gender = "Male" if actor % 2 else "Female"

            # Fake age groups for training
            if actor <= 8:
                age = 25
            elif actor <= 16:
                age = 45
            else:
                age = 65

            file_paths.append(path)
            genders.append(gender)
            emotions.append(emotion)
            ages.append(age)

df = pd.DataFrame({
    "path":file_paths,
    "gender":genders,
    "emotion":emotions,
    "age":ages
})

df.head()

,path,gender,emotion,age
0,C:\Users\yuvra\AGE_GENDER_DETECTION\Datasets\R...,Male,1,25
1,C:\Users\yuvra\AGE_GENDER_DETECTION\Datasets\R...,Male,1,25
2,C:\Users\yuvra\AGE_GENDER_DETECTION\Datasets\R...,Male,1,25
3,C:\Users\yuvra\AGE_GENDER_DETECTION\Datasets\R...,Male,1,25
4,C:\Users\yuvra\AGE_GENDER_DETECTION\Datasets\R...,Male,2,25


**Feature Extraction (MFCC)**

In [5]:
def extract_features(file):

    audio, sr = librosa.load(file, duration=3, offset=0.5)

    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)

    mfcc = np.mean(mfcc.T, axis=0)

    return mfcc

**Create Feature Matrix**

In [6]:
# Generate Feature Dataset
X = []

for file in df["path"]:

    features = extract_features(file)

    X.append(features)

X = np.array(X)

print("Feature Shape:", X.shape)

Feature Shape: (2880, 40)


**Prepare Labels**

In [7]:
# Encode Labels
le_gender = LabelEncoder()
le_emotion = LabelEncoder()

y_gender = le_gender.fit_transform(df["gender"])
y_emotion = le_emotion.fit_transform(df["emotion"])
y_age = df["age"].values

**Train Test Split**

In [9]:
X_train, X_test, y_gender_train, y_gender_test = train_test_split(
    X, y_gender, test_size=0.2, random_state=42
)

_, _, y_age_train, y_age_test = train_test_split(
    X, y_age, test_size=0.2, random_state=42
)

_, _, y_emotion_train, y_emotion_test = train_test_split(
    X, y_emotion, test_size=0.2, random_state=42
)

**Gender Detection Model**

In [10]:
gender_model = Sequential()

gender_model.add(Dense(256,activation='relu',input_shape=(40,)))
gender_model.add(Dropout(0.3))
gender_model.add(Dense(128,activation='relu'))
gender_model.add(Dense(1,activation='sigmoid'))

gender_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

gender_model.fit(
    X_train,
    y_gender_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.6630 - loss: 3.1754 - val_accuracy: 0.8460 - val_loss: 0.5633
Epoch 2/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8258 - loss: 0.9241 - val_accuracy: 0.8460 - val_loss: 0.4750
Epoch 3/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8725 - loss: 0.5218 - val_accuracy: 0.8720 - val_loss: 0.4787
Epoch 4/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9083 - loss: 0.3587 - val_accuracy: 0.9566 - val_loss: 0.1727
Epoch 5/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9343 - loss: 0.2568 - val_accuracy: 0.9761 - val_loss: 0.1231
Epoch 6/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9392 - loss: 0.2646 - val_accuracy: 0.9718 - val_loss: 0.0927
Epoch 7/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9620 - loss: 0.1336 - val_accuracy: 0.9783 - val_loss: 0.0541
Epoch 8/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9577 - loss: 0.1389 - val_accuracy: 0.9848 - val_loss

**Age Prediction Model**

In [11]:
age_model = Sequential()

age_model.add(Dense(256,activation='relu',input_shape=(40,)))
age_model.add(Dense(128,activation='relu'))
age_model.add(Dense(64,activation='relu'))

age_model.add(Dense(1))

age_model.compile(
    optimizer="adam",
    loss="mae"
)

age_model.fit(
    X_train,
    y_age_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 16.2132 - val_loss: 15.1259
Epoch 2/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 14.7475 - val_loss: 14.2096
Epoch 3/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 14.0235 - val_loss: 13.4987
Epoch 4/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 12.9826 - val_loss: 12.2947
Epoch 5/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 12.3948 - val_loss: 11.9423
Epoch 6/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 11.8181 - val_loss: 12.2162
Epoch 7/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 11.6515 - val_loss: 11.9734
Epoch 8/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 10.8766 - val_loss: 10.6280
Epoch 9/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 10.5937 - val_loss: 11.0969
Epoch 10/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9.8893 - val_loss: 10.1937
Epoch 11/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 9.8216 - val_loss: 10.2376
Epoch 12/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - 

**Emotion Detection Model**

In [12]:
y_emotion_cat = to_categorical(y_emotion)

emotion_model = Sequential()

emotion_model.add(Dense(256,activation='relu',input_shape=(40,)))
emotion_model.add(Dropout(0.3))
emotion_model.add(Dense(128,activation='relu'))
emotion_model.add(Dense(len(np.unique(y_emotion)),activation='softmax'))

emotion_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

emotion_model.fit(
    X_train,
    to_categorical(y_emotion_train),
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.1454 - loss: 18.1940 - val_accuracy: 0.2538 - val_loss: 4.0164
Epoch 2/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.1682 - loss: 7.8542 - val_accuracy: 0.2126 - val_loss: 2.5854
Epoch 3/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.1812 - loss: 5.2820 - val_accuracy: 0.2668 - val_loss: 1.9517
Epoch 4/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.1948 - loss: 3.6975 - val_accuracy: 0.2842 - val_loss: 1.9726
Epoch 5/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2154 - loss: 2.8232 - val_accuracy: 0.2950 - val_loss: 2.0206
Epoch 6/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.2203 - loss: 2.4449 - val_accuracy: 0.2907 - val_loss: 1.8207
Epoch 7/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2556 - loss: 2.1761 - val_accuracy: 0.2885 - val_loss: 1.8015
Epoch 8/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.2838 - loss: 2.0124 - val_accuracy: 0.3080 - val_lo

**Save the Models**

In [13]:
models_dir = os.path.join(PROJECT_ROOT,"models")

os.makedirs(models_dir, exist_ok=True)

gender_model.save(os.path.join(models_dir,"voice_gender_model.h5"))
age_model.save(os.path.join(models_dir,"voice_age_model.h5"))
emotion_model.save(os.path.join(models_dir,"emotion_model.h5"))

print("Models saved successfully!")

Models saved successfully!


**Test Prediction**

In [14]:
# Test on Sample Voice
sample = df["path"][0]

features = extract_features(sample)
features = features.reshape(1,-1)

gender = gender_model.predict(features)
age = age_model.predict(features)
emotion = emotion_model.predict(features)

print("Gender:", "Male" if gender>0.5 else "Female")
print("Age:", int(age[0][0]))

emotion_labels = le_emotion.classes_

print("Emotion:", emotion_labels[np.argmax(emotion)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step
Gender: Male
Age: 40
Emotion: 4
